# EV3 Notebook 4: Modelado de Machine Learning
## Bank Marketing Dataset

Entrenamiento y evaluacion de tres modelos de clasificacion para predecir si un cliente suscribira un deposito a plazo.

**Anterior:** `EV3_03_Ingenieria_Caracteristicas.ipynb`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

sns.set_theme(style="whitegrid", palette="muted")
os.makedirs('graficos', exist_ok=True)

# Cargar dataset
df_raw = pd.read_csv('bank-additional-full.csv', sep=';')
df_raw = df_raw.drop_duplicates()
if 'duration' in df_raw.columns:
    df_raw = df_raw.drop(columns=['duration'])

# Renombrar columnas
rename_columns = {
    'y': 'deposito_plazo', 'age': 'edad', 'job': 'trabajo',
    'marital': 'estado_civil', 'education': 'educacion', 'default': 'mora',
    'housing': 'vivienda', 'loan': 'prestamo', 'contact': 'contacto',
    'month': 'mes', 'day_of_week': 'dia_de_la_semana', 'campaign': 'campana',
    'pdays': 'dias_previos', 'previous': 'anterior', 'poutcome': 'resultado_anterior',
    'emp.var.rate': 'var_empleo', 'cons.price.idx': 'indice_precios',
    'cons.conf.idx': 'indice_confianza', 'euribor3m': 'tasa_euribor',
    'nr.employed': 'num_empleados'
}
df_raw = df_raw.rename(columns=rename_columns)
df_raw['deposito_plazo'] = df_raw['deposito_plazo'].map({'yes': 'si', 'no': 'no'})
df_raw['deposito_plazo_num'] = df_raw['deposito_plazo'].map({'si': 1, 'no': 0})
df_ml = df_raw.copy()

# Pipeline completo
cols_unknown = ['trabajo', 'estado_civil', 'educacion', 'mora', 'vivienda', 'prestamo']
for col in cols_unknown:
    df_ml[col] = df_ml[col].replace('unknown', df_ml[df_ml[col] != 'unknown'][col].mode()[0])

def cap_iqr(s):
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    return s.clip(lower=Q1 - 1.5*(Q3-Q1), upper=Q3 + 1.5*(Q3-Q1))

df_ml['edad'] = cap_iqr(df_ml['edad'])
df_ml['campana'] = cap_iqr(df_ml['campana'])

df_ml['es_jubilado'] = (df_ml['trabajo'] == 'retired').astype(int)
df_ml['contactado_antes'] = (df_ml['dias_previos'] != 999).astype(int)
df_ml['contexto_favorable'] = ((df_ml['tasa_euribor'] < 2) & (df_ml['var_empleo'] < 0)).astype(int)
df_ml['intensidad_campana'] = df_ml['campana'] / (df_ml['anterior'] + 1)

bins = [0, 30, 60, 100]
grupo_dummies = pd.get_dummies(
    pd.cut(df_ml['edad'], bins=bins, labels=['Joven', 'Adulto', 'Mayor'], right=True),
    prefix='grupo_edad', drop_first=True)
grupo_dummies.columns = [str(c) for c in grupo_dummies.columns]

cat_cols_ml = df_ml.select_dtypes(include=['object']).columns.drop('deposito_plazo').tolist()
df_ml = pd.get_dummies(df_ml, columns=cat_cols_ml, drop_first=True)
bool_cols = df_ml.select_dtypes(include=['bool']).columns
df_ml[bool_cols] = df_ml[bool_cols].astype(int)

numeric_cols_ml = ['edad', 'campana', 'dias_previos', 'anterior',
                   'var_empleo', 'indice_precios', 'indice_confianza',
                   'tasa_euribor', 'num_empleados']
scaler = StandardScaler()
df_ml[numeric_cols_ml] = scaler.fit_transform(df_ml[numeric_cols_ml])

df_ml = pd.concat([df_ml.reset_index(drop=True), grupo_dummies.reset_index(drop=True)], axis=1)

print(f"Dataset listo: {df_ml.shape[0]:,} filas x {df_ml.shape[1]} columnas")

## Configuracion de modelos y metodologia

**Modelos evaluados:**

| Modelo | Justificacion |
|---|---|
| Regresion Logistica | Baseline interpretable; coeficientes muestran direccion e importancia de cada predictor |
| Random Forest | Robusto a outliers, captura interacciones no lineales, genera importancia de variables |
| XGBoost | Estado del arte en clasificacion tabular; scale_pos_weight compensa el desbalanceo de clases |

**Metodologia: Stratified K-Fold (k=5)**
Se usa validacion cruzada estratificada para respetar la proporcion de clases (90% No / 10% Si) en cada fold.

**Metricas:**

| Metrica | Justificacion |
|---|---|
| ROC-AUC | Principal: mide discriminacion, robusta al desbalanceo |
| Precision | Minimizar llamadas inutiles (falsos positivos) |
| Recall | No perder clientes que si suscribirian (falsos negativos) |
| F1-Score | Balance precision-recall para clases desbalanceadas |

In [ ]:
# Separar features y target
X = df_ml.drop(columns=['deposito_plazo', 'deposito_plazo_num'], errors='ignore')
y = df_ml['deposito_plazo_num']

neg, pos = (y == 0).sum(), (y == 1).sum()
scale_pos = neg / pos

print(f"Features: {X.shape[1]}")
print(f"Muestras: {X.shape[0]:,}")
print(f"Clase 0 (No): {neg:,} ({neg/len(y)*100:.1f}%)")
print(f"Clase 1 (Si): {pos:,} ({pos/len(y)*100:.1f}%)")
print(f"scale_pos_weight para XGBoost: {scale_pos:.2f}")

In [ ]:
# Definicion de modelos
modelos = {
    'Regresion Logistica': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1),
    'XGBoost': xgb.XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=5,
                                   scale_pos_weight=scale_pos, eval_metric='logloss',
                                   random_state=42, n_jobs=-1)
}

scoring = {
    'roc_auc':   'roc_auc',
    'precision': make_scorer(precision_score, zero_division=0),
    'recall':    make_scorer(recall_score, zero_division=0),
    'f1':        make_scorer(f1_score, zero_division=0)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("Modelos configurados. Iniciando validacion cruzada 5-Fold...")

In [ ]:
# Validacion cruzada
resultados = {}
for nombre, modelo in modelos.items():
    print(f"Entrenando {nombre}...", end=' ', flush=True)
    cv_res = cross_validate(modelo, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    resultados[nombre] = {
        'ROC-AUC':   cv_res['test_roc_auc'].mean(),
        'Precision': cv_res['test_precision'].mean(),
        'Recall':    cv_res['test_recall'].mean(),
        'F1-Score':  cv_res['test_f1'].mean(),
        'AUC std':   cv_res['test_roc_auc'].std()
    }
    print(f"AUC = {resultados[nombre]['ROC-AUC']:.4f} +/- {resultados[nombre]['AUC std']:.4f}")

df_res = pd.DataFrame(resultados).T[['ROC-AUC', 'Precision', 'Recall', 'F1-Score', 'AUC std']]
print()
print("Resultados validacion cruzada (5-Fold):")
print(df_res.round(4).to_string())

In [ ]:
# Grafico: comparativa de metricas entre modelos
metricas_plot = ['ROC-AUC', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metricas_plot))
ancho = 0.25
colores_modelo = ['#74b9ff', '#00b894', '#e17055']

fig, ax = plt.subplots(figsize=(12, 6))
for i, (nombre, vals) in enumerate(resultados.items()):
    valores = [vals[m] for m in metricas_plot]
    bars = ax.bar(x + i * ancho, valores, ancho, label=nombre,
                  color=colores_modelo[i], edgecolor='white')
    for bar, val in zip(bars, valores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{val:.3f}", ha='center', va='bottom', fontsize=8, fontweight='bold')
ax.set_xlabel('Metrica de Evaluacion')
ax.set_ylabel('Valor (0-1)')
ax.set_title('Comparativa de Modelos: Regresion Logistica vs Random Forest vs XGBoost')
ax.set_xticks(x + ancho)
ax.set_xticklabels(metricas_plot)
ax.set_ylim(0, 1.12)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax.legend()
sns.despine()
plt.tight_layout()
plt.savefig('graficos/ml_comparativa_metricas_modelos.png', bbox_inches='tight')
plt.show()

In [ ]:
# Grafico: importancia de variables del Random Forest
rf_final = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
rf_final.fit(X, y)

importancias = pd.Series(rf_final.feature_importances_, index=X.columns)
top_20 = importancias.sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 8))
top_20.plot(kind='barh', ax=ax, color=sns.color_palette("Blues_r", len(top_20)), edgecolor='white')
ax.set_xlabel('Importancia (Gini)')
ax.set_title('Top 20 Variables mas Importantes para Predecir Suscripcion (Random Forest)')
ax.invert_yaxis()
sns.despine()
plt.tight_layout()
plt.savefig('graficos/ml_importancia_variables_random_forest.png', bbox_inches='tight')
plt.show()

In [ ]:
# Grafico: matriz de confusion del mejor modelo
mejor = max(resultados, key=lambda k: resultados[k]['ROC-AUC'])
print(f"Mejor modelo: {mejor} (AUC = {resultados[mejor]['ROC-AUC']:.4f})")

modelo_final = modelos[mejor]
modelo_final.fit(X, y)
y_pred = modelo_final.predict(X)

cm = confusion_matrix(y, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No suscribe', 'Si suscribe'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Matriz de Confusion: {mejor}')
plt.tight_layout()
plt.savefig('graficos/ml_matriz_confusion_mejor_modelo.png', bbox_inches='tight')
plt.show()

## Conclusiones del modelado

**Es posible crear un modelo predictivo?**
Si. Los tres modelos superan el baseline aleatorio (AUC > 0.5). XGBoost y Random Forest obtienen los mejores resultados.

**Principales predictores (segun importancia RF):**
- Variables macroeconomicas: tasa_euribor, num_empleados, var_empleo. Confirman que el contexto economico es el factor dominante.
- Historial del cliente: resultado_anterior, dias_previos, anterior. Un cliente exitoso previamente tiene mayor probabilidad de suscribir.
- Variables de campana: a menor numero de contactos, mayor probabilidad de exito.
- Variable creada contexto_favorable aparece con importancia significativa, validando la ingenieria de caracteristicas.

**Variables descartadas:**
- duration (data leaker) eliminada en el preprocesamiento.
- Multicolinealidad controlada con drop_first=True en OHE.

**Contexto economico:**
El modelo sugiere concentrar las campanas en periodos de baja tasa Euribor y variacion negativa del empleo para maximizar la tasa de conversion.